In [1]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

In [2]:
df = pd.read_csv(r'VSRR_Provisional_Drug_Overdose_Death_Counts.csv')

In [3]:
df = df[~df['Indicator'].isin(['Number of Deaths', 'Percent with drugs specified', 'Number of Drug Overdose Deaths'])]

In [4]:
df['Indicator'] = df['Indicator'].replace({
    'Methadone (T40.3)':'Methadone',
    'Natural, semi-synthetic, & synthetic opioids, incl. methadone (T40.2-T40.4)':'Methadone',
    'Natural & semi-synthetic opioids, incl. methadone (T40.2, T40.3)':'Methadone'
})

df['Indicator'] = df['Indicator'].replace({
    'Natural & semi-synthetic opioids (T40.2)':'Opioids',
    'Opioids (T40.0-T40.4,T40.6)':'Opioids',
    'Synthetic opioids, excl. methadone (T40.4)':'Opioids'
})

df['Indicator'] = df['Indicator'].replace({
    'Cocaine (T40.5)':'Cocaine',
    'Heroin (T40.1)':'Heroin',
    'Psychostimulants with abuse potential (T43.6)':'Psychostimulants'
})

In [5]:
df.drop(['Data Value', 'Predicted Value', 'Percent Complete', 'Footnote Symbol', 'Period'], axis=1, inplace=True)

In [6]:
df['Month'] = pd.to_datetime(df['Month'], format='%B').dt.month

In [7]:
opioid_classes = ['Heroin', 'Methadone', 'Opioids']

df['Opioid Binary'] = df['Indicator'].isin(opioid_classes).astype(int)

In [8]:
df[['Indicator', 'Opioid Binary']].drop_duplicates().sort_values('Indicator')

,Indicator,Opioid Binary
0,Cocaine,0
124,Heroin,1
248,Methadone,1
372,Opioids,1
1240,Psychostimulants,0


In [9]:
X = df.drop(columns = ['Indicator', 'Opioid Binary'])
y = df['Opioid Binary']

In [10]:
cat_cols = X.select_dtypes(include='object').columns.tolist()
num_cols = X.select_dtypes(exclude='object').columns.tolist()

In [19]:
preprocessor = ColumnTransformer(
    transformers = [
        ('cat', OneHotEncoder(), cat_cols),
        ('num', StandardScaler(), num_cols)
    ]
)

hist_preprocessor = ColumnTransformer(
    transformers=[
    ('num', StandardScaler(), num_cols)
    ],
    remainder = 'drop'
)

In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [16]:
rf_model = RandomForestClassifier(
    random_state=42
)

xgbc_model = XGBClassifier(
    random_state=42
)

hist_model = HistGradientBoostingClassifier(
    random_state=42
)

In [21]:
rf_pipeline = Pipeline ([
    ('preprocessor', preprocessor),
    ('model', rf_model)
])

xgbc_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', xgbc_model)
])

hist_pipeline = Pipeline([
    ('preprocessor', hist_preprocessor),
    ('model', hist_model)
])

In [ ]:
rf_pipeline.fit(X_train, y_train)

xgbc_pipeline.fit(X_train, y_train)

hist_pipeline.fit(X_train, y_train)

In [ ]:
rf_y_pred = rf_pipeline.predict(X_test)
rf_y_proba = rf_pipeline.predict_proba(X_test)[:,1]

xgbc_y_pred = xgbc_pipeline.predict(X_test)
xgbc_y_proba = xgbc_pipeline.predict_proba(X_test)[:, 1]

hist_y_pred = hist_pipeline.predict(X_test)
hist_y_proba = hist_pipeline.predict(X_test)[:,1]

In [ ]:
print(classification_report(y_test, rf_y_pred))
print("ROC AUC:", roc_auc_score(y_test, rf_y_proba))

print(classification_report(y_test, xgbc_y_pred))
print("ROC AUC:", roc_auc_score (y_test, xgbc_y_proba))

print(classfification_report(y_test, hist_y_pred))
print("ROC AUC:", roc_auc_score(y_test, hist_y_proba))

              precision    recall  f1-score   support

           0       0.17      0.01      0.02      2584
           1       0.78      0.98      0.87      9023

    accuracy                           0.77     11607
   macro avg       0.47      0.50      0.45     11607
weighted avg       0.64      0.77      0.68     11607

ROC AUC: 0.14880725778531576
              precision    recall  f1-score   support

           0       0.61      0.01      0.02      2584
           1       0.78      1.00      0.87      9023

    accuracy                           0.78     11607
   macro avg       0.69      0.50      0.45     11607
weighted avg       0.74      0.78      0.69     11607

ROC AUC: 0.2639191930906535
